In [0]:
import os
import re
import requests
from zipfile import ZipFile
from bs4 import BeautifulSoup
from pathlib import Path
from io import BytesIO

In [0]:
%sql
USE CATALOG dados_fa;
USE SCHEMA pessoal;
CREATE VOLUME IF NOT EXISTS xml_files;

In [0]:
MESES = [
    "Janeiro", "Fevereiro", "Marco", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
]

BASE_URL = "https://in.gov.br/acesso-a-informacao/dados-abertos/base-de-dados"

OUTPUT_FILE = Path("../documentos/dou_zips.txt")
XML_DIR = Path("/Volumes/dados_fa/pessoal/xml_files")

EXPRESSAO_ALVO = "Ministério da Defesa"


def extrair_links_zip(ano, mes, session=None):
    session = session or requests.Session()

    resp = session.get(BASE_URL, params={"ano": ano, "mes": mes}, timeout=60)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    links = {
        a["href"]
        for a in soup.find_all("a", href=True)
        if ".zip" in a["href"].lower()
    }
    return sorted(links)


def carregar_links_existentes(path: Path):
    if not path.exists():
        return set()

    with open(path, "r", encoding="utf-8") as f:
        return {linha.strip() for linha in f if linha.strip()}


def salvar_link_processado(path: Path, link: str):
    with open(path, "a", encoding="utf-8") as f:
        f.write(link + "\n")


def obter_nome_zip(link: str):
    match = re.search(r"(S\d{8}\.zip)", link, flags=re.IGNORECASE)
    return match.group(1) if match else None


def decodificar_bytes_xml(data: bytes) -> str:
    # tenta utf-8 primeiro; se falhar, latin-1 costuma funcionar para XMLs antigos do DOU
    for encoding in ("utf-8", "latin-1"):
        try:
            return data.decode(encoding)
        except UnicodeDecodeError:
            continue
    return data.decode("utf-8", errors="ignore")


def baixar_zip_em_memoria(session, url):
    resp = session.get(url, timeout=120)
    resp.raise_for_status()
    return BytesIO(resp.content)


def processar_zip_em_memoria(zip_bytes, nome_zip: str, xml_base_dir: Path, expressao: str):
    pasta_saida = xml_base_dir / Path(nome_zip).stem
    pasta_saida.mkdir(parents=True, exist_ok=True)

    salvos = 0

    with ZipFile(zip_bytes) as zf:
        for member in zf.infolist():
            if member.is_dir():
                continue
            if not member.filename.lower().endswith(".xml"):
                continue

            with zf.open(member) as xml_file:
                xml_bytes = xml_file.read()

            texto = decodificar_bytes_xml(xml_bytes)

            if expressao in texto:
                destino_xml = pasta_saida / Path(member.filename).name
                with open(destino_xml, "wb") as f:
                    f.write(xml_bytes)
                salvos += 1

    # remove pasta vazia se nenhum xml foi salvo
    if salvos == 0:
        try:
            pasta_saida.rmdir()
        except OSError:
            pass

    return salvos


def main():
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    XML_DIR.mkdir(parents=True, exist_ok=True)

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "pt-BR,pt;q=0.9",
    })

    links_existentes = carregar_links_existentes(OUTPUT_FILE)

    total_links_novos = 0
    total_xmls_salvos = 0

    for ano in range(2002, 2027):
        for mes in MESES:
            try:
                links = extrair_links_zip(ano, mes, session=session)
                novos = [link for link in links if link not in links_existentes]

                xmls_mes = 0

                for link in novos:
                    nome_zip = obter_nome_zip(link)
                    if not nome_zip:
                        print(f"Nome não identificado no link: {link}")
                        continue

                    try:
                        zip_bytes = baixar_zip_em_memoria(session, link)
                        salvos = processar_zip_em_memoria(
                            zip_bytes=zip_bytes,
                            nome_zip=nome_zip,
                            xml_base_dir=XML_DIR,
                            expressao=EXPRESSAO_ALVO
                        )

                        salvar_link_processado(OUTPUT_FILE, link)
                        links_existentes.add(link)

                        total_links_novos += 1
                        total_xmls_salvos += salvos
                        xmls_mes += salvos

                    except Exception as e:
                        print(f"Erro ao processar ZIP {nome_zip}: {e}")

                print(f"{ano} - {mes}: {len(links)} encontrados, {len(novos)} novos, {xmls_mes} XMLs salvos")

            except Exception as e:
                print(f"Erro em {ano} - {mes}: {e}")

    print(f"\nTotal de novos links salvos: {total_links_novos}")
    print(f"Total de XMLs salvos: {total_xmls_salvos}")
    print(f"Arquivo de links: {OUTPUT_FILE}")
    print(f"Diretório dos XMLs: {XML_DIR}")


main()

In [0]:
%sql
DROP VOLUME zip_files;